# Задача 2: Предобработка и редактирование текста

Данный ноутбук реализует пайплайн очистки данных для классификации обращений в приемную комиссию. 

**Этапы работы:**
1. Настройка окружения и импорты.
2. Демонстрация работы функций на одном тестовом примере.
3. Запуск полного пайплайна (очистка + лемматизация + аугментация).
4. Сохранение результатов в `train.parquet` и `test.parquet`.
5. Проверка метрик F1 (сравнение различных методов обработки).

In [10]:
# ==========================================
# БЛОК 1: НАСТРОЙКА ОКРУЖЕНИЯ И ИМПОРТЫ
# ==========================================
import sys
import os
import pandas as pd

# Поднимаемся из папки notebooks в корень проекта, чтобы Python увидел папку src
sys.path.append(os.path.abspath('..'))

# Импортируем модуль предобработки
from src.preprocessing import (
    clean_text, 
    fix_typos, 
    lemmatize_text, 
    preprocess_dataframe, 
    split_and_save_data
)

# Настройка отображения таблиц (чтобы длинный текст не обрезался многоточием)
pd.set_option('display.max_colwidth', None)

print("Импорт прошел успешно!")
print("Pandas версия:", pd.__version__)

Импорт прошел успешно!
Pandas версия: 3.1.1


## 1. Проверка работы функций на тестовом примере
Убедимся, что регулярные выражения и словари работают корректно, до запуска на всех 588 письмах.

In [11]:
# ==========================================
# БЛОК 2: ТЕСТ ФУНКЦИЙ НА ОДНОМ ПРИМЕРЕ
# ==========================================
test_text = "Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим."

print("=== 1. ИСХОДНЫЙ ТЕКСТ ===")
print(test_text)

# Шаг 1: Базовая очистка (удаление тегов, приветствий, подписей)
cleaned = clean_text(test_text)
print("\n=== 2. ПОСЛЕ clean_text (очистка) ===")
print(cleaned)

# Шаг 2: Исправление опечаток по словарю
fixed = fix_typos(cleaned)
print("\n=== 3. ПОСЛЕ fix_typos (исправление опечаток) ===")
print(fixed)

# Шаг 3: Лемматизация (приведение к начальной форме)
lemmatized = lemmatize_text(fixed)
print("\n=== 4. ПОСЛЕ lemmatize_text (лемматизация) ===")
print(lemmatized)

# Ожидаемый результат: текст без тегов, подписей, с исправленной опечаткой и приведенный к инфинитивам.

=== 1. ИСХОДНЫЙ ТЕКСТ ===
Здравствуйте!<br>не хватило # балла по биологии. Может ли она пересдать экзамен по билогии?<br><br>С уважением, Максим.

=== 2. ПОСЛЕ clean_text (очистка) ===
не хватило <num> балла по биологии может ли она пересдать экзамен по билогии

=== 3. ПОСЛЕ fix_typos (исправление опечаток) ===
не хватило <num> балла по биологии может ли она пересдать экзамен по биология

=== 4. ПОСЛЕ lemmatize_text (лемматизация) ===
хватить <num> балл биология пересдать экзамен биология


## 2. Запуск полного пайплайна на всем датасете
Здесь мы загружаем исходный файл и применяем все методы: очистку, лемматизацию, стемминг и аугментацию редких классов.

In [6]:
# ==========================================
# БЛОК 3: ЗАГРУЗКА ДАННЫХ И ПАЙПЛАЙН
# ==========================================
# 1. Загружаем исходные данные
df = pd.read_excel('../data/mails.xlsx')
print(f"Загружено строк: {df.shape[0]}")

# 2. Запускаем полный пайплайн (очистка -> опечатки -> лемматизация -> стемминг -> сленг -> аугментация)
df_processed = preprocess_dataframe(df)

# 3. Выводим примеры «До/После» обработки
print("\n=== ПРИМЕРЫ ОБРАБОТКИ ===")
display(df_processed[['CONTENT', 'CONTENT_clean', 'CONTENT_lemm', 'CONTENT_stem']].head(3))

Загружено строк: 588
1. Применяем базовую очистку...
2. Исправляем опечатки...
3. Применяем лемматизацию...
4. Применяем стемминг (для классики)...
5. Применяем замену сленга...
Выполняем аугментацию редких классов...
Размер датасета до аугментации: 588
Размер датасета после аугментации: 738

=== ПРИМЕРЫ ОБРАБОТКИ ===


,CONTENT,CONTENT_clean,CONTENT_lemm,CONTENT_stem
0,"Здравствуйте! Поступила в ваш институт на бюджет Педиатрического факультета. Написала отказ от зачисления, как удостоверится в том, что меня отчислили? С уважнием, Ксения.",поступила в ваш институт на бюджет педиатрического факультета написала отказ от зачисления как удостоверится в том что меня отчислили,поступить ваш институт бюджет педиатрический факультет написать отказ зачисление удостовериться отчислить,посту ваш инсти бюдже педиа факул напис отказ зачис удост отчис
1,"Здравствуйте! Я прошла конкурс по трем дисциплинам, зачислена дисциплину по #-й приоритету, возможно ли перейти сейчас на другой приоритет(управление персоналом) или ""поезд ушел""?",я прошла конкурс по трем дисциплинам зачислена дисциплину по <num> й приоритету возможно ли перейти сейчас на другой приоритет управление персоналом или поезд ушел,пройти конкурс три дисциплина зачислить дисциплина <num> приоритет возможно перейти приоритет управление персонал поезд уйти,пройт конку три дисци зачис дисци <num> приор возмо перей приор управ персо поезд уйти
2,"Здравствуйте! Скажите пожалуйста могу ли я подать документы на направление Экология если в этом году ЕГЭ в ## классе Не сдавала есть результаты года за # класс русского языка, математики, и география.",скажите пожалуйста могу ли я подать документы на направление экология если в этом году егэ в <num> классе не сдавала есть результаты года за <num> класс русского языка математики и география,сказать пожалуйста мочь подать документ направление экология год егэ <num> класс сдавать результат год <num> класс русский язык математика география,сказа пожал мочь подат докум напра эколо год егэ <num> класс сдава резул год <num> класс русск язык матем геогр


## 3. Сохранение выборок
После запуска этой ячейки файлы `train.parquet` и `test.parquet` будут перезаписаны. 

In [8]:
# ==========================================
# БЛОК 4: СОХРАНЕНИЕ TRAIN/TEST
# ==========================================
train_df, test_df = split_and_save_data(df_processed)

# Проверяем, что файлы сохранились корректно
print("\n=== ПРОВЕРКА СОХРАНЕННЫХ ФАЙЛОВ ===")
train_check = pd.read_parquet('../data/train.parquet')
test_check = pd.read_parquet('../data/test.parquet')

print(f"Train: {train_check.shape[0]} строк, {train_check.shape[1]} колонок")
print(f"Test:  {test_check.shape[0]} строк, {test_check.shape[1]} колонок")
print(f"Колонки: {train_check.columns.tolist()}")


6. Делим данные на Train/Test (0.9/0.1)...
Train сохранен: 664 строк
Test сохранен:  74 строк

=== ПРОВЕРКА СОХРАНЕННЫХ ФАЙЛОВ ===
Train: 664 строк, 7 колонок
Test:  74 строк, 7 колонок
Колонки: ['CONTENT', 'TYPE_HOTLINE', 'class', 'CONTENT_clean', 'CONTENT_lemm', 'CONTENT_stem', 'CONTENT_edited']


## 4. Оценка качества (Метрики F1)
Обучаем простую модель (Logistic Regression + TF-IDF), чтобы доказать эффективность предобработки и аугментации.

In [9]:
# ==========================================
# БЛОК 5: ЭКСПЕРИМЕНТ С МЕТРИКАМИ
# ==========================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

def evaluate_f1(text_column):
    """
    Обучает LogReg на указанной колонке и возвращает Macro/Weighted F1.
    """
    # 1. Векторизация текста
    vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
    X_train = vectorizer.fit_transform(train_check[text_column])
    X_test = vectorizer.transform(test_check[text_column])
    
    # 2. Обучение модели (class_weight='balanced' важен для дисбаланса классов)
    clf = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
    clf.fit(X_train, train_check['class'])
    
    # 3. Предсказание и подсчет метрик
    y_pred = clf.predict(X_test)
    macro = f1_score(test_check['class'], y_pred, average='macro')
    weighted = f1_score(test_check['class'], y_pred, average='weighted')
    return macro, weighted

print("--- РЕЗУЛЬТАТЫ ДЛЯ ОТЧЕТА ---")

# 1. Только очистка (без нормализации)
m_clean, w_clean = evaluate_f1('CONTENT_clean')
print(f"1. Очистка:          Macro F1 = {m_clean:.4f} | Weighted = {w_clean:.4f}")

# 2. Очистка + Лемматизация
m_lemm, w_lemm = evaluate_f1('CONTENT_lemm')
print(f"2. Лемматизация:     Macro F1 = {m_lemm:.4f} | Weighted = {w_lemm:.4f}")

# 3. Очистка + Стемминг (мягкая нормализация)
m_stem, w_stem = evaluate_f1('CONTENT_stem')
print(f"3. Стемминг:         Macro F1 = {m_stem:.4f} | Weighted = {w_stem:.4f}")

# 4. Очистка + Лемматизация + Замена сленга
m_edit, w_edit = evaluate_f1('CONTENT_edited')
print(f"4. Сленг:            Macro F1 = {m_edit:.4f} | Weighted = {w_edit:.4f}")

--- РЕЗУЛЬТАТЫ ДЛЯ ОТЧЕТА ---
1. Очистка:          Macro F1 = 0.8602 | Weighted = 0.8893
2. Лемматизация:     Macro F1 = 0.8741 | Weighted = 0.9164
3. Стемминг:         Macro F1 = 0.8741 | Weighted = 0.9164
4. Сленг:            Macro F1 = 0.8741 | Weighted = 0.9164


## Итоги

- Пайплайн успешно реализован в `src/preprocessing.py`.
- Применена аугментация для редких классов (Аспирантура, Регистрация, Достижения), что позволило искусственно увеличить выборку с 588 до 738 записей.
- Лучшие результаты показали методы, включающие нормализацию текста (лемматизация или стемминг).
- Данные сохранены в `data/train.parquet` и `data/test.parquet` и готовы к использованию в нейросетевых моделях.